# Phase 4 - Modeling: Car Price Prediction

Notebook ini menjalankan Phase 4 dari PRD: membuat model `LinearRegression` dan melakukan training pada `X_train` dan `y_train` hasil Phase 3.

Input yang digunakan:
- `outputs/phase3/X_train.csv`
- `outputs/phase3/y_train.csv`

Output utama:
- Model Linear Regression yang sudah dilatih
- File model: `outputs/phase4/car_price_linear_regression_model.pkl`
- Ringkasan model: `outputs/phase4/model_summary.json`

## 1. Import Library

Library utama yang digunakan adalah `pandas` untuk membaca data, `LinearRegression` dari `scikit-learn` untuk modeling, dan `joblib` untuk menyimpan model.

In [ ]:
from pathlib import Path
import json

import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## 2. Load Data Training

Data training berasal dari Phase 3 Data Preparation, sehingga missing value sudah ditangani dan data sudah dipisahkan 80% training dan 20% testing.

In [ ]:
BASE_DIR = Path.cwd()
PHASE3_DIR = BASE_DIR / "outputs" / "phase3"
PHASE4_DIR = BASE_DIR / "outputs" / "phase4"
PHASE4_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(PHASE3_DIR / "X_train.csv")
X_test = pd.read_csv(PHASE3_DIR / "X_test.csv")
y_train = pd.read_csv(PHASE3_DIR / "y_train.csv")["Price_in_thousands"]
y_test = pd.read_csv(PHASE3_DIR / "y_test.csv")["Price_in_thousands"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
X_train.head()

## 3. Membuat Model Regresi

Sesuai ketentuan project, model yang digunakan adalah `LinearRegression`.

In [ ]:
model = LinearRegression()
model

## 4. Training Model

Model dilatih menggunakan `X_train` sebagai variabel independen dan `y_train` sebagai variabel dependen.

In [ ]:
model.fit(X_train, y_train)

print("Training selesai.")
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

## 5. Interpretasi Koefisien

Koefisien menunjukkan arah dan besar hubungan linear antara setiap fitur dan harga mobil dalam ribuan USD, dengan asumsi fitur lain konstan.

In [ ]:
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_,
}).sort_values("coefficient", ascending=False)

coef_df

## 6. Simpan Model

Model disimpan sebagai `.pkl` agar dapat digunakan kembali pada tahap evaluasi, prediksi, atau deployment.

In [ ]:
model_path = PHASE4_DIR / "car_price_linear_regression_model.pkl"
summary_path = PHASE4_DIR / "model_summary.json"

joblib.dump(model, model_path)

model_summary = {
    "model_type": "LinearRegression",
    "target": "Price_in_thousands",
    "features": X_train.columns.tolist(),
    "train_rows": int(len(X_train)),
    "intercept": float(model.intercept_),
    "coefficients": {feature: float(value) for feature, value in zip(X_train.columns, model.coef_)},
    "model_path": str(model_path),
}

summary_path.write_text(json.dumps(model_summary, indent=2), encoding="utf-8")

print("Model saved to:", model_path)
print("Summary saved to:", summary_path)

## 7. Evaluasi Model dengan RMSE dan R2 Score

Model dievaluasi menggunakan data testing dari Phase 3. RMSE menunjukkan rata-rata besar error prediksi dalam satuan ribuan USD, sedangkan R2 Score menunjukkan seberapa besar variasi harga yang dapat dijelaskan oleh model.

In [ ]:
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

evaluation_metrics = {
    "rmse": float(rmse),
    "r2_score": float(r2),
    "test_rows": int(len(X_test)),
}

print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

## 8. Scatter Plot Actual vs Predicted

Scatter plot digunakan untuk melihat kedekatan hasil prediksi terhadap nilai aktual. Titik yang semakin dekat dengan garis diagonal menunjukkan prediksi yang semakin baik.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.75, edgecolor="black")

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], color="red", linewidth=2)

plt.title("Actual vs Predicted Car Price")
plt.xlabel("Actual Price (in thousands USD)")
plt.ylabel("Predicted Price (in thousands USD)")
plt.grid(True, alpha=0.3)
plt.tight_layout()

scatter_plot_path = PHASE4_DIR / "actual_vs_predicted_scatter.png"
plt.savefig(scatter_plot_path, dpi=150)
plt.show()

print("Scatter plot saved to:", scatter_plot_path)

## 9. Simpan Hasil Evaluasi

Metrik evaluasi disimpan agar dapat dipakai kembali pada laporan final atau deployment readiness.

In [ ]:
evaluation_path = PHASE4_DIR / "evaluation_metrics.json"
evaluation_metrics["scatter_plot_path"] = str(scatter_plot_path)
evaluation_path.write_text(json.dumps(evaluation_metrics, indent=2), encoding="utf-8")

print("Evaluation metrics saved to:", evaluation_path)

## Phase 4 Checklist

- Model `LinearRegression` berhasil dibuat.
- Model berhasil dilatih pada `X_train` dan `y_train`.
- RMSE dan R2 Score berhasil dihitung menggunakan data testing.
- Scatter plot actual vs predicted berhasil dibuat.
- Model disimpan untuk digunakan pada Phase 5 Evaluation dan Phase 6 Deployment.